# 📊 Helios Logistics — Interactive Staffing Analysis\n## DC Rhein-Main: Recommendations vs Actuals, Volumes, Correlations & Decision Log\n\n---

In [15]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    pio.templates.default = "plotly_white"
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "nbformat", "ipywidgets", "-q"])
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    pio.templates.default = "plotly_white"

# ── Load data ──
present = pd.read_csv("data/clean/present_long.csv", parse_dates=["date"])
recs = pd.read_csv("data/clean/recommendations_long.csv", parse_dates=["decision_date", "planned_week_start", "date"])
volumes = pd.read_csv("data/clean/volumes_long.csv", parse_dates=["date"])

with open("data/decision_log.json") as f:
    decision_log = json.load(f)

with open("data/cost_model.json") as f:
    cost_model = json.load(f)

# ── Derived columns ──
present["dow"] = present["date"].dt.day_name()
present["week"] = present["date"].dt.isocalendar().week.astype(int)
present["month"] = present["date"].dt.to_period("M").astype(str)

# Aggregate recommendations: total operative per day
rec_operative = recs[recs["group"] == "operative"].groupby("date")["recommended_person_days"].sum().reset_index()
rec_operative.columns = ["date", "rec_operative"]

# Merge all into one master dataframe
df = present.merge(rec_operative, on="date", how="inner")
df = df.merge(volumes, on="date", how="inner")

# Error columns
df["error"] = df["rec_operative"] - df["present_operative_person_days"]
df["error_pct"] = df["error"] / df["present_operative_person_days"] * 100
df["total_volume"] = df["picks_realized"] + df["outbound_realized"] + df["inbound_realized"]

# Pick-by-light regime flag
df["regime"] = np.where(df["date"] >= "2026-08-24", "Post Pick-by-Light", "Pre Pick-by-Light")

print(f"✅ Loaded {len(df)} working days | {len(recs)} activity-level recommendations | {len(decision_log['entries'])} decision log notes")
print(f"   Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

✅ Loaded 98 working days | 1862 activity-level recommendations | 15 decision log notes
   Date range: 2026-05-18 → 2026-10-02


,date,present_total_person_days,present_operative_person_days,dow,week,month,rec_operative,picks_forecast,picks_realized,outbound_forecast,outbound_realized,inbound_forecast,inbound_realized,error,error_pct,total_volume,regime
0,2026-05-18,64.25,56.25,Monday,21,2026-05,64.7,9555,9587,1229,1228,1766,1850,8.45,15.022222,12665,Pre Pick-by-Light
1,2026-05-19,62.25,54.25,Tuesday,21,2026-05,66.5,10774,10297,1382,1362,1608,1618,12.25,22.580645,13277,Pre Pick-by-Light
2,2026-05-20,58.75,50.75,Wednesday,21,2026-05,59.7,8098,8234,1384,1362,1404,1470,8.95,17.635468,11066,Pre Pick-by-Light
3,2026-05-21,61.50,53.50,Thursday,21,2026-05,65.9,9653,10010,1558,1548,1579,1516,12.40,23.177570,13074,Pre Pick-by-Light
4,2026-05-22,62.25,54.25,Friday,21,2026-05,64.0,9274,9155,1712,1631,1321,1377,9.75,17.972350,12163,Pre Pick-by-Light


## 1. Recommended vs Actual Staffing — Daily Timeline\n\nThe optimiser's recommendation (blue) vs what was actually needed (green), with the gap shaded. Decision log notes are annotated as vertical markers.

In [16]:
# ── 1. Recommended vs Actual with Decision Log annotations ──
fig = go.Figure()

# Shaded area for error band
fig.add_trace(go.Scatter(
    x=pd.concat([df["date"], df["date"][::-1]]),
    y=pd.concat([df["rec_operative"], df["present_operative_person_days"][::-1]]),
    fill="toself", fillcolor="rgba(255,99,71,0.15)", line=dict(width=0),
    name="Overstaffing Gap", hoverinfo="skip", showlegend=True
))

fig.add_trace(go.Scatter(
    x=df["date"], y=df["rec_operative"],
    mode="lines+markers", name="Recommended (Optimiser)",
    line=dict(color="#636EFA", width=2), marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=df["date"], y=df["present_operative_person_days"],
    mode="lines+markers", name="Actual Operative",
    line=dict(color="#00CC96", width=2), marker=dict(size=4)
))

# Annotate decision log notes as vertical lines
colors_log = {"Maya": "#EF553B", "Jonas": "#AB63FA", "Selin": "#FFA15A"}
for entry in decision_log["entries"]:
    dt = pd.to_datetime(entry["captured_on"])
    fig.add_vline(x=dt, line_dash="dot", line_color=colors_log.get(entry["author"], "gray"), opacity=0.5)
    fig.add_annotation(x=dt, y=df["rec_operative"].max() + 1.5, text=entry["id"],
                       showarrow=False, font=dict(size=8, color=colors_log.get(entry["author"], "gray")),
                       yanchor="bottom")

# Pick-by-light line
fig.add_vline(x=pd.to_datetime("2026-08-24"), line_dash="dash", line_color="red", line_width=2)
fig.add_annotation(x=pd.to_datetime("2026-08-24"), y=df["rec_operative"].max() + 3,
                   text="⚡ Pick-by-Light Live", showarrow=False,
                   font=dict(size=10, color="red", family="Arial Black"))

fig.update_layout(
    title="Daily Staffing: Recommended vs Actual Operative Person-Days",
    xaxis_title="Date", yaxis_title="Person-Days",
    height=500, legend=dict(orientation="h", y=-0.15),
    hovermode="x unified"
)
fig.show()

## 2. Optimiser Error Over Time\n\nThe daily error (Recommended − Actual) and its rolling 5-day average. Error is **always positive** — the optimiser overstaffs every single day.

In [17]:
# ── 2. Error timeline with rolling average ──
df["error_rolling5"] = df["error"].rolling(5, min_periods=1).mean()

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=df["date"], y=df["error"],
    name="Daily Error", marker_color=np.where(df["regime"] == "Pre Pick-by-Light", "#636EFA", "#EF553B"),
    opacity=0.6, hovertemplate="Date: %{x}<br>Error: %{y:.1f} person-days<extra></extra>"
))

fig2.add_trace(go.Scatter(
    x=df["date"], y=df["error_rolling5"],
    mode="lines", name="5-Day Rolling Avg",
    line=dict(color="black", width=2, dash="dot")
))

fig2.add_hline(y=df["error"].mean(), line_dash="dash", line_color="gray",
               annotation_text=f"Mean: +{df['error'].mean():.1f}", annotation_position="top left")

fig2.add_vline(x=pd.to_datetime("2026-08-24"), line_dash="dash", line_color="red", line_width=2)

fig2.update_layout(
    title="Optimiser Error (Recommended − Actual) — Always Positive = Always Overstaffs",
    xaxis_title="Date", yaxis_title="Error (Person-Days)",
    height=400, legend=dict(orientation="h", y=-0.15),
    hovermode="x unified"
)
fig2.show()

## 3. Day-of-Week Patterns\n\nActual staffing, recommendations, and error broken down by day of week. **Wednesdays** are consistently the lowest-staffed day.

In [18]:
# ── 3. Day-of-week box plots (Actual, Recommended, Error) ──
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

fig3 = make_subplots(rows=1, cols=3, subplot_titles=[
    "Actual Operative", "Recommended", "Error (Rec − Actual)"
], horizontal_spacing=0.08)

fig3.add_trace(go.Box(
    x=df["dow"], y=df["present_operative_person_days"],
    name="Actual", marker_color="#00CC96", boxmean=True
), row=1, col=1)

fig3.add_trace(go.Box(
    x=df["dow"], y=df["rec_operative"],
    name="Recommended", marker_color="#636EFA", boxmean=True
), row=1, col=2)

fig3.add_trace(go.Box(
    x=df["dow"], y=df["error"],
    name="Error", marker_color="#EF553B", boxmean=True
), row=1, col=3)

for i in range(1, 4):
    fig3.update_xaxes(categoryorder="array", categoryarray=dow_order, row=1, col=i)

fig3.update_layout(
    title="Day-of-Week Distribution: Actual vs Recommended vs Error",
    height=450, showlegend=False
)
fig3.show()

## 4. Volume Analysis — Forecast vs Realized\n\nPicks, outbound pallets, and inbound pallets: comparing what was forecasted vs what actually happened. Volume forecasts are accurate — the problem is NOT the demand forecast.

In [19]:
# ── 4. Volume: Forecast vs Realized (3 panels) ──
vol_types = [
    ("picks_forecast", "picks_realized", "Picks"),
    ("outbound_forecast", "outbound_realized", "Outbound Pallets"),
    ("inbound_forecast", "inbound_realized", "Inbound Pallets"),
]

fig4 = make_subplots(rows=3, cols=1, subplot_titles=[v[2] for v in vol_types],
                     shared_xaxes=True, vertical_spacing=0.08)

for i, (fc, rl, label) in enumerate(vol_types, 1):
    fig4.add_trace(go.Scatter(
        x=df["date"], y=df[fc], mode="lines", name=f"{label} Forecast",
        line=dict(dash="dot", width=1.5), legendgroup=label, showlegend=True
    ), row=i, col=1)
    fig4.add_trace(go.Scatter(
        x=df["date"], y=df[rl], mode="lines", name=f"{label} Realized",
        line=dict(width=2), legendgroup=label, showlegend=True
    ), row=i, col=1)

fig4.update_layout(
    title="Volume: Forecast vs Realized — Forecasts Are Accurate",
    height=700, legend=dict(orientation="h", y=-0.05),
    hovermode="x unified"
)
fig4.show()

## 5. Volume vs Staffing Correlations\n\nScatter plots showing how realized volumes (picks, outbound, inbound, total) correlate with actual operative staffing. Color-coded by pre/post pick-by-light regime.

In [20]:
# ── 5. Correlation scatter plots ──
vol_cols = [
    ("picks_realized", "Picks Realized"),
    ("outbound_realized", "Outbound Realized"),
    ("inbound_realized", "Inbound Realized"),
    ("total_volume", "Total Volume"),
]

fig5 = make_subplots(rows=2, cols=2, subplot_titles=[v[1] for v in vol_cols])

regime_colors = {"Pre Pick-by-Light": "#636EFA", "Post Pick-by-Light": "#EF553B"}

for idx, (col, label) in enumerate(vol_cols):
    row, col_n = divmod(idx, 2)
    for regime_name, color in regime_colors.items():
        mask = df["regime"] == regime_name
        r_val = df.loc[mask, col].corr(df.loc[mask, "present_operative_person_days"])
        fig5.add_trace(go.Scatter(
            x=df.loc[mask, col], y=df.loc[mask, "present_operative_person_days"],
            mode="markers", name=f"{regime_name} (r={r_val:.2f})",
            marker=dict(color=color, size=6, opacity=0.7),
            legendgroup=regime_name, showlegend=(idx == 0),
            hovertemplate=f"{label}: %{{x}}<br>Operative: %{{y:.1f}}<extra>{regime_name}</extra>"
        ), row=row+1, col=col_n+1)
    
    # Overall trendline
    z = np.polyfit(df[col], df["present_operative_person_days"], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 50)
    fig5.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line), mode="lines",
        line=dict(color="gray", dash="dash", width=1),
        showlegend=False, hoverinfo="skip"
    ), row=row+1, col=col_n+1)

fig5.update_layout(
    title="Volume vs Actual Staffing — Correlation by Regime",
    height=600, legend=dict(orientation="h", y=-0.08)
)
fig5.show()

# Print correlation matrix
print("\\n📈 Correlation Matrix: Volumes vs Actual Operative Staffing")
corr_cols = ["present_operative_person_days", "picks_realized", "outbound_realized", "inbound_realized", "total_volume", "rec_operative", "error"]
print(df[corr_cols].corr().round(3).to_string())

\n📈 Correlation Matrix: Volumes vs Actual Operative Staffing
                               present_operative_person_days  picks_realized  outbound_realized  inbound_realized  total_volume  rec_operative  error
present_operative_person_days                          1.000           0.748              0.447             0.464         0.804          0.848 -0.001
picks_realized                                         0.748           1.000              0.223             0.507         0.988          0.869  0.443
outbound_realized                                      0.447           0.223              1.000            -0.352         0.329          0.458  0.149
inbound_realized                                       0.464           0.507             -0.352             1.000         0.524          0.504  0.207
total_volume                                           0.804           0.988              0.329             0.524         1.000          0.918  0.444
rec_operative                          

## 6. Correlation Heatmap\n\nFull correlation matrix across all key variables — staffing, volumes, errors.

In [21]:
# ── 6. Correlation heatmap ──
corr_labels = {
    "present_operative_person_days": "Actual Operative",
    "rec_operative": "Recommended",
    "error": "Error (Rec−Act)",
    "picks_realized": "Picks",
    "outbound_realized": "Outbound",
    "inbound_realized": "Inbound",
    "total_volume": "Total Volume",
    "picks_forecast": "Picks Forecast",
}

corr_df = df[list(corr_labels.keys())].rename(columns=corr_labels)
corr_matrix = corr_df.corr().round(3)

fig6 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns, y=corr_matrix.columns,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=corr_matrix.values.round(2), texttemplate="%{text}",
    textfont=dict(size=10),
    hovertemplate="Row: %{y}<br>Col: %{x}<br>r = %{z:.3f}<extra></extra>"
))

fig6.update_layout(
    title="Correlation Heatmap: Staffing, Volumes & Errors",
    height=550, width=700,
    xaxis=dict(tickangle=45)
)
fig6.show()

## 7. Activity-Level Recommendations Breakdown\n\nHow the optimiser distributes person-days across the 15 operative activities. Interactive stacked area chart over time.

In [22]:
# ── 7. Activity-level stacked area ──
rec_op = recs[recs["group"] == "operative"].copy()
rec_pivot = rec_op.pivot_table(index="date", columns="activity", values="recommended_person_days", aggfunc="sum").fillna(0)

# Sort by mean (largest activities on bottom)
activity_order = rec_pivot.mean().sort_values(ascending=False).index.tolist()

fig7 = go.Figure()
for act in reversed(activity_order):
    fig7.add_trace(go.Scatter(
        x=rec_pivot.index, y=rec_pivot[act],
        mode="lines", stackgroup="one", name=act,
        hovertemplate=f"{act}: %{{y:.1f}}<extra></extra>"
    ))

# Overlay actual operative as a line
fig7.add_trace(go.Scatter(
    x=df["date"], y=df["present_operative_person_days"],
    mode="lines", name="Actual Operative Total",
    line=dict(color="black", width=3, dash="dot"),
    hovertemplate="Actual: %{y:.1f}<extra></extra>"
))

fig7.add_vline(x=pd.to_datetime("2026-08-24"), line_dash="dash", line_color="red", line_width=2)

fig7.update_layout(
    title="Activity-Level Recommendations (Stacked) vs Actual Total",
    xaxis_title="Date", yaxis_title="Person-Days",
    height=550, legend=dict(font=dict(size=9)),
    hovermode="x unified"
)
fig7.show()

## 8. Activity Variability Analysis\n\nWhich activities are fixed (low CV) vs variable (high CV)? Fixed activities (transit, team leads, etc.) should be hard-coded; variable ones (picking, putaway, staging) are where optimization matters.

In [23]:
# ── 8. Activity variability (CV bar chart) ──
activity_stats = rec_pivot.describe().T[["mean", "std"]].copy()
activity_stats["cv_pct"] = (activity_stats["std"] / activity_stats["mean"] * 100).round(1)
activity_stats = activity_stats.sort_values("cv_pct", ascending=True)

fig8 = make_subplots(rows=1, cols=2, subplot_titles=["Mean Person-Days/Day", "Coefficient of Variation (%)"],
                     horizontal_spacing=0.15)

fig8.add_trace(go.Bar(
    y=activity_stats.index, x=activity_stats["mean"],
    orientation="h", marker_color="#636EFA", name="Mean",
    hovertemplate="%{y}: %{x:.1f} person-days<extra></extra>"
), row=1, col=1)

fig8.add_trace(go.Bar(
    y=activity_stats.index, x=activity_stats["cv_pct"],
    orientation="h",
    marker_color=np.where(activity_stats["cv_pct"] < 2, "#00CC96", "#EF553B"),
    name="CV%",
    hovertemplate="%{y}: %{x:.1f}% CV<extra></extra>"
), row=1, col=2)

fig8.update_layout(
    title="Activity Variability: Fixed (green) vs Variable (red)",
    height=500, showlegend=False
)
fig8.show()

print("\\n📋 Activity Statistics:")
print(activity_stats[["mean", "std", "cv_pct"]].rename(columns={"mean": "Mean PD/Day", "std": "Std Dev", "cv_pct": "CV%"}).to_string())

\n📋 Activity Statistics:
                            Mean PD/Day   Std Dev   CV%
activity                                               
Aisle maintenance              1.000000  0.000000   0.0
Pick QA                        1.000000  0.000000   0.0
Returns / QC                   1.000000  0.000000   0.0
Team leads                     4.000000  0.000000   0.0
VNA replenishment              1.000000  0.000000   0.0
Yard shunting                  2.000000  0.000000   0.0
Transit drivers                4.021429  0.045972   1.1
Co-Packing line                4.023469  0.051375   1.3
Picking                       12.812245  1.237829   9.7
Replenishment / relocation     2.716327  0.265817   9.8
Putaway                       11.603061  1.257264  10.8
Receiving                      2.655102  0.289729  10.9
Unloading                      2.284694  0.255800  11.2
Staging                        8.079592  1.121207  13.9
Loading                        5.768367  0.808537  14.0


## 9. Pre vs Post Pick-by-Light Regime Comparison\n\nThe pick-by-light deployment on Aug 24 changed everything. Comparing error distributions and staffing patterns before and after.

In [24]:
# ── 9. Pre vs Post pick-by-light comparison ──
fig9 = make_subplots(rows=1, cols=3, subplot_titles=[
    "Actual Operative", "Error (Rec − Actual)", "Error % Distribution"
], horizontal_spacing=0.08)

for regime_name, color in regime_colors.items():
    mask = df["regime"] == regime_name
    fig9.add_trace(go.Box(
        y=df.loc[mask, "present_operative_person_days"], name=regime_name,
        marker_color=color, boxmean=True, legendgroup=regime_name, showlegend=True
    ), row=1, col=1)
    fig9.add_trace(go.Box(
        y=df.loc[mask, "error"], name=regime_name,
        marker_color=color, boxmean=True, legendgroup=regime_name, showlegend=False
    ), row=1, col=2)
    fig9.add_trace(go.Histogram(
        x=df.loc[mask, "error_pct"], name=regime_name,
        marker_color=color, opacity=0.6, legendgroup=regime_name, showlegend=False,
        nbinsx=15
    ), row=1, col=3)

fig9.update_layout(
    title="Regime Comparison: Pre vs Post Pick-by-Light (Aug 24)",
    height=400, legend=dict(orientation="h", y=-0.15)
)
fig9.show()

# Stats table
regime_stats = df.groupby("regime").agg(
    days=("date", "count"),
    actual_mean=("present_operative_person_days", "mean"),
    rec_mean=("rec_operative", "mean"),
    error_mean=("error", "mean"),
    error_pct_mean=("error_pct", "mean"),
).round(2)
print("\\n📊 Regime Statistics:")
print(regime_stats.to_string())

\n📊 Regime Statistics:
                    days  actual_mean  rec_mean  error_mean  error_pct_mean
regime                                                                     
Post Pick-by-Light    30        53.85     65.66       11.81           21.96
Pre Pick-by-Light     68        53.43     63.22        9.79           18.39


## 10. Decision Log Validation Dashboard\n\nEach planner note (L01–L15) mapped against the data. Shows which claims are confirmed, stale, or disputed.

In [25]:
# ── 10. Decision log validation dashboard ──
log_entries = decision_log["entries"]
validations = {
    "L01": {"verdict": "Likely True", "confidence": 75, "detail": "Transit rec varies 4.0–4.2; actual likely always 4"},
    "L02": {"verdict": "Confirmed", "confidence": 90, "detail": "Co-Packing fixed at 4 (reconfirmed by L10)"},
    "L03": {"verdict": "Stale", "confidence": 30, "detail": "Superseded by L11/L12 (pick-by-light: −25-27%)"},
    "L04": {"verdict": "Plausible", "confidence": 55, "detail": "Monday inbound is highest; staffing uplift marginal"},
    "L05": {"verdict": "Unverified", "confidence": 25, "detail": "Only ~2 data points; anecdotal"},
    "L06": {"verdict": "Plausible", "confidence": 60, "detail": "Low-volume days may need fewer pickers"},
    "L07": {"verdict": "Plausible", "confidence": 60, "detail": "Post-closure backlog likely but data sparse"},
    "L08": {"verdict": "Partially Confirmed", "confidence": 70, "detail": "Jul–Aug actuals did drop; data supports cut"},
    "L09": {"verdict": "Disputed", "confidence": 35, "detail": "Contradicts L08; data favours L08"},
    "L10": {"verdict": "Confirmed", "confidence": 95, "detail": "8 weeks of Co-Packing at exactly 4"},
    "L11": {"verdict": "Confirmed", "confidence": 90, "detail": "Pick-by-light: picking need −25% confirmed"},
    "L12": {"verdict": "Confirmed", "confidence": 92, "detail": "Two planners confirm −25-27%; error widened post Aug 24"},
    "L13": {"verdict": "Insufficient Data", "confidence": 30, "detail": "Only ~3 days with inbound >2000"},
    "L14": {"verdict": "Confirmed", "confidence": 85, "detail": "Sep outbound climbing; Oct mean 59.4"},
    "L15": {"verdict": "Confirmed", "confidence": 85, "detail": "Autumn ramp confirmed; picks +8% vs prior month"},
}

# Build dataframe for plotting
log_df = pd.DataFrame([
    {
        "id": e["id"],
        "date": e["captured_on"],
        "author": e["author"],
        "scope": str(e["scope"]) if isinstance(e["scope"], list) else e["scope"],
        "note_preview": e["note"][:80] + "...",
        "verdict": validations[e["id"]]["verdict"],
        "confidence": validations[e["id"]]["confidence"],
        "detail": validations[e["id"]]["detail"],
    }
    for e in log_entries
])

verdict_colors = {
    "Confirmed": "#00CC96", "Likely True": "#72B7B2", "Partially Confirmed": "#FECB52",
    "Plausible": "#FFA15A", "Stale": "#EF553B", "Disputed": "#FF6692",
    "Unverified": "#B6E880", "Insufficient Data": "#AB63FA"
}

fig10 = go.Figure()

fig10.add_trace(go.Bar(
    x=log_df["id"], y=log_df["confidence"],
    marker_color=[verdict_colors.get(v, "gray") for v in log_df["verdict"]],
    text=log_df["verdict"], textposition="outside",
    customdata=np.stack([log_df["author"], log_df["scope"], log_df["detail"], log_df["note_preview"]], axis=-1),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Author: %{customdata[0]}<br>"
        "Scope: %{customdata[1]}<br>"
        "Verdict: %{text}<br>"
        "Confidence: %{y}%<br>"
        "Detail: %{customdata[2]}<br>"
        "Note: %{customdata[3]}<extra></extra>"
    )
))

fig10.add_hline(y=70, line_dash="dash", line_color="green", annotation_text="Actionable threshold (70%)")
fig10.add_hline(y=40, line_dash="dash", line_color="red", annotation_text="Unreliable (<40%)")

fig10.update_layout(
    title="Decision Log Validation: Confidence Scores by Entry (Hover for Details)",
    xaxis_title="Log Entry", yaxis_title="Confidence %",
    yaxis=dict(range=[0, 105]),
    height=450
)
fig10.show()

## 11. Cost Impact Analysis\n\nEstimated daily cost of overstaffing under the asymmetric cost model. Overstaffing costs €230/person-day; understaffing costs €41.40 (under tolerance) or €600 (over tolerance).

In [26]:
# ── 11. Cost impact analysis ──
def compute_daily_cost(row):
    """Asymmetric cost per the cost model."""
    excess = row["error"]  # rec - actual (always positive here = overstaffing)
    if excess >= 0:
        return excess * 230  # overstaffing: idle cost
    else:
        shortfall = abs(excess)
        base = shortfall * 41.4  # overtime premium (18% of 230)
        if shortfall > 2.0:
            base += (shortfall - 2.0) * 600  # SLA penalty
        return base

df["daily_cost"] = df.apply(compute_daily_cost, axis=1)
df["cumulative_cost"] = df["daily_cost"].cumsum()

# What if we applied a flat trim?
df["trimmed_rec"] = df["rec_operative"] * (1 - 0.163)
df["trimmed_error"] = df["trimmed_rec"] - df["present_operative_person_days"]
df["trimmed_cost"] = df.apply(
    lambda r: max(0, r["trimmed_error"]) * 230 + (
        abs(min(0, r["trimmed_error"])) * 41.4 +
        max(0, abs(min(0, r["trimmed_error"])) - 2.0) * 600
    ), axis=1
)
df["trimmed_cumulative"] = df["trimmed_cost"].cumsum()

fig11 = go.Figure()

fig11.add_trace(go.Scatter(
    x=df["date"], y=df["cumulative_cost"],
    mode="lines", name="Baseline (Stale Rate Card)",
    line=dict(color="#EF553B", width=2.5),
    fill="tozeroy", fillcolor="rgba(239,85,59,0.1)",
    hovertemplate="Date: %{x}<br>Cumulative: €%{y:,.0f}<extra>Baseline</extra>"
))

fig11.add_trace(go.Scatter(
    x=df["date"], y=df["trimmed_cumulative"],
    mode="lines", name="After −16.3% Flat Trim",
    line=dict(color="#00CC96", width=2.5),
    fill="tozeroy", fillcolor="rgba(0,204,150,0.1)",
    hovertemplate="Date: %{x}<br>Cumulative: €%{y:,.0f}<extra>Trimmed</extra>"
))

savings = df["cumulative_cost"].iloc[-1] - df["trimmed_cumulative"].iloc[-1]
fig11.add_annotation(
    x=df["date"].iloc[-1], y=df["cumulative_cost"].iloc[-1],
    text=f"Total savings: €{savings:,.0f}",
    showarrow=True, arrowhead=2, ax=-100, ay=-30,
    font=dict(size=12, color="#00CC96")
)

fig11.update_layout(
    title="Cumulative Cost: Baseline vs −16.3% Flat Trim Correction",
    xaxis_title="Date", yaxis_title="Cumulative Cost (€)",
    height=450, legend=dict(orientation="h", y=-0.15),
    hovermode="x unified"
)
fig11.show()

print(f"\\n💰 Total baseline cost: €{df['cumulative_cost'].iloc[-1]:,.0f}")
print(f"💰 Total after trim:    €{df['trimmed_cumulative'].iloc[-1]:,.0f}")
print(f"💰 Savings:             €{savings:,.0f} ({savings/df['cumulative_cost'].iloc[-1]*100:.1f}%)")

\n💰 Total baseline cost: €234,600
💰 Total after trim:    €19,871
💰 Savings:             €214,729 (91.5%)


## 12. Weekly Error Trend\n\nWeekly aggregated error showing the structural pattern: steady overstaffing that worsens after pick-by-light (week 35+).

In [27]:
# ── 12. Weekly error trend ──
weekly = df.groupby("week").agg(
    error_mean=("error", "mean"),
    error_sum=("error", "sum"),
    actual_mean=("present_operative_person_days", "mean"),
    rec_mean=("rec_operative", "mean"),
    cost_sum=("daily_cost", "sum"),
    days=("date", "count"),
).reset_index()

fig12 = make_subplots(rows=2, cols=1, subplot_titles=[
    "Weekly Mean Error (Person-Days)", "Weekly Total Cost (€)"
], shared_xaxes=True, vertical_spacing=0.12)

fig12.add_trace(go.Bar(
    x=weekly["week"], y=weekly["error_mean"],
    name="Avg Error",
    marker_color=np.where(weekly["week"] >= 35, "#EF553B", "#636EFA"),
    hovertemplate="Week %{x}: %{y:.1f} PD<extra></extra>"
), row=1, col=1)

fig12.add_trace(go.Bar(
    x=weekly["week"], y=weekly["cost_sum"],
    name="Weekly Cost",
    marker_color=np.where(weekly["week"] >= 35, "#EF553B", "#636EFA"),
    hovertemplate="Week %{x}: €%{y:,.0f}<extra></extra>"
), row=2, col=1)

fig12.add_vline(x=34.5, line_dash="dash", line_color="red", line_width=2)
fig12.add_annotation(x=34.5, y=weekly["error_mean"].max(),
                     text="Pick-by-Light", showarrow=False,
                     font=dict(color="red", size=10), row=1, col=1)

fig12.update_layout(
    title="Weekly Error & Cost Trend — Error Worsens After Pick-by-Light",
    height=500, showlegend=False
)
fig12.update_xaxes(title_text="ISO Week", row=2, col=1)
fig12.show()

## 13. Volume vs Error — Does More Volume = More Error?\n\nTesting whether the optimiser's error scales with volume or is a fixed bias.

In [28]:
# ── 13. Volume vs Error scatter (is error proportional to volume?) ──
fig13 = px.scatter(
    df, x="total_volume", y="error", color="regime",
    color_discrete_map=regime_colors,
    trendline="ols", trendline_scope="overall",
    size="daily_cost", size_max=15,
    hover_data={"date": True, "error_pct": ":.1f", "daily_cost": ":,.0f"},
    labels={
        "total_volume": "Total Realized Volume (picks+outbound+inbound)",
        "error": "Error (Person-Days)",
        "regime": "Regime",
        "daily_cost": "Daily Cost (€)"
    },
    title="Volume vs Error — Bubble Size = Daily Cost"
)
fig13.update_layout(height=450)
fig13.show()

# Correlation test
from scipy import stats
r, p = stats.pearsonr(df["total_volume"], df["error"])
print(f"\\n📈 Pearson r(total_volume, error) = {r:.3f}, p = {p:.4f}")
if p < 0.05:
    print(f"   → Significant correlation: error {'increases' if r > 0 else 'decreases'} with volume")
else:
    print(f"   → No significant correlation: error is a fixed bias, not volume-dependent")

ModuleNotFoundError: No module named 'statsmodels'

## 14. Forecast Accuracy Analysis\n\nHow accurate are the volume forecasts? The bottleneck is NOT forecasting — it's the rate card conversion.

In [29]:
# ── 14. Forecast accuracy ──
df["picks_err_pct"] = (df["picks_forecast"] - df["picks_realized"]) / df["picks_realized"] * 100
df["outbound_err_pct"] = (df["outbound_forecast"] - df["outbound_realized"]) / df["outbound_realized"] * 100
df["inbound_err_pct"] = (df["inbound_forecast"] - df["inbound_realized"]) / df["inbound_realized"] * 100

fig14 = make_subplots(rows=1, cols=3, subplot_titles=["Picks Forecast Error %", "Outbound Forecast Error %", "Inbound Forecast Error %"])

for i, col in enumerate(["picks_err_pct", "outbound_err_pct", "inbound_err_pct"], 1):
    fig14.add_trace(go.Histogram(
        x=df[col], nbinsx=25,
        marker_color="#636EFA", opacity=0.7,
        hovertemplate="Error: %{x:.1f}%<br>Count: %{y}<extra></extra>"
    ), row=1, col=i)
    fig14.add_vline(x=0, line_dash="dash", line_color="gray", row=1, col=i)
    fig14.add_vline(x=df[col].mean(), line_dash="solid", line_color="red", row=1, col=i)

fig14.update_layout(
    title="Volume Forecast Error Distributions — Centered Near Zero = Accurate",
    height=350, showlegend=False
)
fig14.show()

print("📊 Forecast Accuracy Summary:")
for name, col in [("Picks", "picks_err_pct"), ("Outbound", "outbound_err_pct"), ("Inbound", "inbound_err_pct")]:
    print(f"   {name}: Mean error = {df[col].mean():.2f}%, Std = {df[col].std():.2f}%, MAPE = {df[col].abs().mean():.2f}%")

📊 Forecast Accuracy Summary:
   Picks: Mean error = 0.08%, Std = 3.44%, MAPE = 2.70%
   Outbound: Mean error = 0.18%, Std = 3.72%, MAPE = 3.02%
   Inbound: Mean error = 0.66%, Std = 3.97%, MAPE = 3.19%


## 15. Volume by Day-of-Week\n\nAre there consistent daily volume patterns that should inform staffing? How do picks, outbound, and inbound vary across weekdays?

In [30]:
# ── 15. Volume by day-of-week ──
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

fig15 = make_subplots(rows=1, cols=3, subplot_titles=["Picks Realized", "Outbound Realized", "Inbound Realized"])

for i, col in enumerate(["picks_realized", "outbound_realized", "inbound_realized"], 1):
    fig15.add_trace(go.Box(
        x=df["dow"], y=df[col], boxmean=True,
        marker_color=["#636EFA", "#EF553B", "#00CC96"][i-1],
    ), row=1, col=i)
    fig15.update_xaxes(categoryorder="array", categoryarray=dow_order, row=1, col=i)

fig15.update_layout(
    title="Volume by Day-of-Week — Monday Inbound Spike Supports L04 (Receiving +1 on Mon)",
    height=400, showlegend=False
)
fig15.show()

# Print DoW volume means
dow_vol = df.groupby("dow")[["picks_realized", "outbound_realized", "inbound_realized"]].mean().reindex(dow_order).round(0)
print("\\n📊 Mean Volume by Day-of-Week:")
print(dow_vol.to_string())

\n📊 Mean Volume by Day-of-Week:
           picks_realized  outbound_realized  inbound_realized
dow                                                           
Monday             9618.0             1225.0            1696.0
Tuesday           10429.0             1419.0            1571.0
Wednesday          8108.0             1389.0            1356.0
Thursday           9876.0             1510.0            1478.0
Friday             9542.0             1790.0            1351.0


## 16. Summary Statistics & Key Takeaways

In [31]:
# ── 16. Summary statistics ──
print("=" * 70)
print("  HELIOS LOGISTICS — DC RHEIN-MAIN — KEY METRICS SUMMARY")
print("=" * 70)

print(f"\n📅 Training period:  {df['date'].min().date()} → {df['date'].max().date()} ({len(df)} working days)")
print(f"\n👥 STAFFING:")
print(f"   Actual operative mean:      {df['present_operative_person_days'].mean():.1f} person-days/day")
print(f"   Recommended operative mean: {df['rec_operative'].mean():.1f} person-days/day")
print(f"   Mean overstaffing:          +{df['error'].mean():.1f} person-days/day (+{df['error_pct'].mean():.1f}%)")
print(f"   Days overstaffed:           {(df['error'] > 0).sum()}/{len(df)} ({(df['error'] > 0).mean()*100:.0f}%)")

print(f"\n⚡ PICK-BY-LIGHT IMPACT:")
pre = df[df["regime"] == "Pre Pick-by-Light"]
post = df[df["regime"] == "Post Pick-by-Light"]
print(f"   Pre  (n={len(pre)}): error = +{pre['error'].mean():.1f} PD ({pre['error_pct'].mean():.1f}%)")
print(f"   Post (n={len(post)}): error = +{post['error'].mean():.1f} PD ({post['error_pct'].mean():.1f}%)")
print(f"   → Error widened by +{post['error'].mean() - pre['error'].mean():.1f} PD after pick-by-light")

print(f"\n💰 COST IMPACT:")
print(f"   Total baseline cost:     €{df['daily_cost'].sum():,.0f}")
print(f"   Mean daily cost:         €{df['daily_cost'].mean():,.0f}")
print(f"   After −16.3% trim:      €{df['trimmed_cost'].sum():,.0f}")
print(f"   Savings:                 €{df['daily_cost'].sum() - df['trimmed_cost'].sum():,.0f} ({(1 - df['trimmed_cost'].sum()/df['daily_cost'].sum())*100:.1f}%)")

print(f"\n📈 VOLUME FORECAST ACCURACY:")
for name, col in [("Picks", "picks_err_pct"), ("Outbound", "outbound_err_pct"), ("Inbound", "inbound_err_pct")]:
    print(f"   {name}: MAPE = {df[col].abs().mean():.2f}% — {'✅ Excellent' if df[col].abs().mean() < 5 else '⚠️ Moderate'}")

print(f"\n📝 DECISION LOG:")
confirmed = sum(1 for v in validations.values() if v["verdict"] in ["Confirmed", "Likely True"])
stale = sum(1 for v in validations.values() if v["verdict"] in ["Stale", "Disputed"])
print(f"   Total notes:     {len(validations)}")
print(f"   Confirmed/Likely: {confirmed} ({confirmed/len(validations)*100:.0f}%)")
print(f"   Stale/Disputed:   {stale} ({stale/len(validations)*100:.0f}%)")

print(f"\n🔑 KEY CORRELATIONS:")
key_corrs = [
    ("Picks ↔ Actual Staffing", df["picks_realized"].corr(df["present_operative_person_days"])),
    ("Total Volume ↔ Actual", df["total_volume"].corr(df["present_operative_person_days"])),
    ("Total Volume ↔ Error", df["total_volume"].corr(df["error"])),
    ("Recommended ↔ Actual", df["rec_operative"].corr(df["present_operative_person_days"])),
]
for label, r in key_corrs:
    strength = "strong" if abs(r) > 0.6 else "moderate" if abs(r) > 0.3 else "weak"
    print(f"   {label}: r = {r:.3f} ({strength})")

print("\n" + "=" * 70)

  HELIOS LOGISTICS — DC RHEIN-MAIN — KEY METRICS SUMMARY

📅 Training period:  2026-05-18 → 2026-10-02 (98 working days)

👥 STAFFING:
   Actual operative mean:      53.6 person-days/day
   Recommended operative mean: 64.0 person-days/day
   Mean overstaffing:          +10.4 person-days/day (+19.5%)
   Days overstaffed:           98/98 (100%)

⚡ PICK-BY-LIGHT IMPACT:
   Pre  (n=68): error = +9.8 PD (18.4%)
   Post (n=30): error = +11.8 PD (22.0%)
   → Error widened by +2.0 PD after pick-by-light

💰 COST IMPACT:
   Total baseline cost:     €234,600
   Mean daily cost:         €2,394
   After −16.3% trim:      €19,871
   Savings:                 €214,729 (91.5%)

📈 VOLUME FORECAST ACCURACY:
   Picks: MAPE = 2.70% — ✅ Excellent
   Outbound: MAPE = 3.02% — ✅ Excellent
   Inbound: MAPE = 3.19% — ✅ Excellent

📝 DECISION LOG:
   Total notes:     15
   Confirmed/Likely: 7 (47%)
   Stale/Disputed:   2 (13%)

🔑 KEY CORRELATIONS:
   Picks ↔ Actual Staffing: r = 0.748 (strong)
   Total Volume ↔ Actu